# Milestone 6: Causal Early-Warning & Alert Engine Validation

This notebook validates the newly implemented causal alert engine for both India periodic data and Maharashtra continuous telemetry.

**Key Scientific Constraints Enforced:**
- Strict backward-looking causal logic (no future information leaked).
- Separate methodologies tailored for periodic vs telemetry data.
- Data-quality gating preventing erroneous alerts.
- Explict handling of large telemetry gaps.
- Descriptive severity states (not hydrological laws).


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append("../src")
from alerting.config import AlertConfig
from alerting.engine import generate_india_alerts, generate_maharashtra_alerts
from features.seasonal import assign_india_season

DATA_DIR = Path("../data")
INDIA_NORMALIZED = DATA_DIR / "processed/india/india_normalized.parquet"
MH_NORMALIZED = DATA_DIR / "processed/maharashtra/maharashtra_normalized.parquet"

df_in_raw = pd.read_parquet(INDIA_NORMALIZED)
df_mh_raw = pd.read_parquet(MH_NORMALIZED)


## 1. India Periodic Alerts (Causal Seasonal Z-Score)
Alerts generated using strictly prior historical means and standard deviations. Applies a deterministic state machine for operational events.

In [2]:
config = AlertConfig()
df_in = assign_india_season(df_in_raw)
alerts_india = generate_india_alerts(df_in, config)

print("Alert Event State Distribution (India):")
display(alerts_india["alert_event_state"].value_counts())

print("\nAlert Evidence Example (India - WARNING/CRITICAL):")
mask = alerts_india["alert_event_state"].isin([config.STATE_WARNING, config.STATE_CRITICAL])
display(alerts_india[mask][[
    "candidate_site_id", "observation_date", "analytical_season", 
    "water_level_raw", "historical_reference", "deviation", "historical_std", 
    "standardized_score", "historical_sample_count", 
    "instantaneous_state", "alert_event_state"
]].head(10))


Alert Event State Distribution (India):


alert_event_state
NORMAL                            708255
INSUFFICIENT_HISTORY              540701
WATCH                             136128
CRITICAL_STATISTICAL_DEVIATION     93468
WARNING                            63896
DATA_QUALITY_EVENT                 26720
Name: count, dtype: int64


Alert Evidence Example (India - WARNING/CRITICAL):


,candidate_site_id,observation_date,analytical_season,water_level_raw,historical_reference,deviation,historical_std,standardized_score,historical_sample_count,instantaneous_state,alert_event_state
26,IND_00026bd84267,2019-11-28,POST_MONSOON,1.0,1.166667,-0.166667,0.051640,-3.227486,6,WARNING,WARNING
27,IND_00026bd84267,2020-01-28,RABI_WINTER,1.9,1.366667,0.533333,0.206559,2.581989,6,WATCH,WARNING
28,IND_00026bd84267,2020-11-05,POST_MONSOON,0.9,1.142857,-0.242857,0.078680,-3.086660,7,WARNING,WARNING
29,IND_00026bd84267,2021-01-01,RABI_WINTER,1.2,1.442857,-0.242857,0.276026,-0.879834,7,NORMAL,WARNING
30,IND_00026bd84267,2021-08-01,MONSOON,0.9,1.342857,-0.442857,0.512696,-0.863781,7,NORMAL,WARNING
39,IND_00026bd84267,2023-11-01,POST_MONSOON,1.8,1.060000,0.740000,0.150555,4.915163,10,CRITICAL_STATISTICAL_DEVIATION,CRITICAL_STATISTICAL_DEVIATION
40,IND_00026bd84267,2024-01-01,RABI_WINTER,2.0,1.400000,0.600000,0.319722,1.876630,10,NORMAL,CRITICAL_STATISTICAL_DEVIATION
41,IND_00026bd84267,2024-05-10,PRE_MONSOON,2.0,1.531111,0.468889,0.639598,0.733099,9,NORMAL,CRITICAL_STATISTICAL_DEVIATION
206,IND_00093552a94a,2010-11-01,POST_MONSOON,3.1,5.172727,-2.072727,0.670956,-3.089215,11,WARNING,WARNING
207,IND_00093552a94a,2011-01-01,RABI_WINTER,4.6,5.781818,-1.181818,0.815865,-1.448546,11,NORMAL,WARNING


## 2. Maharashtra Telemetry Alerts (Normalized Causal Residual)
Alerts based on strictly backward-looking persistence deviations, normalized by robust historical rolling variance, with explicit gap handling.

In [3]:
alerts_mh = generate_maharashtra_alerts(df_mh_raw, config)

print("Alert Event State Distribution (Maharashtra):")
display(alerts_mh["alert_event_state"].value_counts())

print("\nAlert Evidence Example (Maharashtra - WATCH/WARNING/CRITICAL):")
mask_mh = alerts_mh["alert_event_state"].isin([config.STATE_WATCH, config.STATE_WARNING, config.STATE_CRITICAL])
display(alerts_mh[mask_mh][[
    "station_name", "observation_timestamp", 
    "water_level_raw", "historical_reference", "deviation", "historical_std", "standardized_score",
    "time_since_previous_obs_hours", "instantaneous_state", "alert_event_state"
]].head(10))


Alert Event State Distribution (Maharashtra):


alert_event_state
NORMAL                            158020
INSUFFICIENT_HISTORY               25401
SEQUENCE_GAPPED                    12068
WATCH                               9780
DATA_QUALITY_EVENT                  8121
CRITICAL_STATISTICAL_DEVIATION      2779
WARNING                             2622
Name: count, dtype: int64


Alert Evidence Example (Maharashtra - WATCH/WARNING/CRITICAL):


,station_name,observation_timestamp,water_level_raw,historical_reference,deviation,historical_std,standardized_score,time_since_previous_obs_hours,instantaneous_state,alert_event_state
52,AALEGAON,2026-01-24T00:00:00,-12.661,-13.886,1.225,0.484526,2.528245,6.0,WATCH,WATCH
53,AALEGAON,2026-01-24T06:00:00,-12.628,-12.661,0.033,0.516529,0.063888,6.0,NORMAL,WATCH
54,AALEGAON,2026-01-24T12:00:00,-12.551,-12.628,0.077,0.512020,0.150385,6.0,NORMAL,WATCH
56,AALEGAON,2026-01-26T00:00:00,-13.614,-12.535,-1.079,0.503535,-2.142848,30.0,WATCH,WATCH
57,AALEGAON,2026-01-26T06:00:00,-13.595,-13.614,0.019,0.515150,0.036882,6.0,NORMAL,WATCH
58,AALEGAON,2026-01-26T12:00:00,-14.105,-13.595,-0.510,0.510945,-0.998150,6.0,NORMAL,WATCH
60,AALEGAON,2026-01-27T00:00:00,-11.836,-14.174,2.338,0.504526,4.634051,6.0,CRITICAL_STATISTICAL_DEVIATION,CRITICAL_STATISTICAL_DEVIATION
61,AALEGAON,2026-01-27T06:00:00,-11.772,-11.836,0.064,0.593719,0.107795,6.0,NORMAL,CRITICAL_STATISTICAL_DEVIATION
62,AALEGAON,2026-01-27T12:00:00,-11.770,-11.772,0.002,0.589115,0.003395,6.0,NORMAL,CRITICAL_STATISTICAL_DEVIATION
76,AALEGAON,2026-02-01T00:00:00,-12.579,-14.148,1.569,0.558580,2.808907,6.0,WATCH,WATCH


## 3. Data Quality Gating Check
Confirming that rows with severe data quality problems are gated into DATA_QUALITY_EVENT.

In [4]:
gated_in = alerts_india[alerts_india["alert_event_state"] == config.STATE_DATA_QUALITY]
print(f"India Quality Gated Rows: {len(gated_in)}")
display(gated_in[["water_level_raw", "quality_flags", "instantaneous_state", "alert_event_state"]].head(3))

gated_mh = alerts_mh[alerts_mh["alert_event_state"] == config.STATE_DATA_QUALITY]
print(f"\nMaharashtra Quality Gated Rows: {len(gated_mh)}")
display(gated_mh[["water_level_raw", "quality_flags", "instantaneous_state", "alert_event_state"]].head(3))


India Quality Gated Rows: 26720


,water_level_raw,quality_flags,instantaneous_state,alert_event_state
904,10.6,CONFLICTING_DUPLICATE;LABEL_PROVENANCE_UNVERIFIED,DATA_QUALITY_EVENT,DATA_QUALITY_EVENT
905,8.7,CONFLICTING_DUPLICATE;LABEL_PROVENANCE_UNVERIFIED,DATA_QUALITY_EVENT,DATA_QUALITY_EVENT
909,9.3,CONFLICTING_DUPLICATE;LABEL_PROVENANCE_UNVERIFIED,DATA_QUALITY_EVENT,DATA_QUALITY_EVENT



Maharashtra Quality Gated Rows: 8121


,water_level_raw,quality_flags,instantaneous_state,alert_event_state
2086,-7.084,CONFLICTING_DUPLICATE,DATA_QUALITY_EVENT,DATA_QUALITY_EVENT
2087,-8.802,CONFLICTING_DUPLICATE,DATA_QUALITY_EVENT,DATA_QUALITY_EVENT
2088,-7.585,CONFLICTING_DUPLICATE,DATA_QUALITY_EVENT,DATA_QUALITY_EVENT
